# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadQasimTahir/flyrank_Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method Choice: Random Forest Classifier.  

Why it fits this lane: In Week 2, we saw that a single Decision Tree can learn basic rules but is prone to ties and rigid boundaries. Random Forest builds an ensemble of trees, providing a much smoother, continuous probability score that is ideal for stack-ranking a review queue. It effortlessly handles non-linear interactions (e.g., how days_since_last_update interacts with avg_position and ctr) without requiring strict feature scaling, making it a robust choice for scoring content opportunities.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split Design: Client-Grouped Holdout (GroupShuffleSplit).  

Why this is honest: Pages belonging to the same client are not statistically independent. They share domain authority, template structures, industry seasonality, and content strategies. If we use a standard random train/test split, pages from "Client A" will appear in both the training and test sets, allowing the model to memorize client-specific quirks rather than learning universal search signals. Grouping by client_id ensures that the model is tested on entirely unseen clients, proving that its ranking logic generalizes to new data.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

We will train the Random Forest on the same observable features and compare its Precision@50 directly against the Week-4 baseline (stale_visible score) on the exact same unseen test clients.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import sys
import subprocess
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# 1. Environment Setup (
if "google.colab" in sys.modules:
    REPO_DIR = "flyrank-ml-internship-starter"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

# 2. Loading  the starter slice & define label
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

# 3. Creating the Week 4 Baseline Score
df["baseline_score"] = (df["days_since_last_update"] >= 180).astype(int) * df["impressions_90d"]

# 4. Grouped Split (by client_id)
features = ["impressions_90d", "days_since_last_update", "avg_position", "ctr", "word_count", "content_age_days"]
X = df[features]
y = df["is_declining_label"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]
baseline_test = df["baseline_score"].iloc[test_idx]

# 5. Training the Random Forest
rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(n_estimators=100, max_depth=5, class_weight="balanced", random_state=42))
])
rf_pipeline.fit(X_train, y_train)

# 6. Predicting Probabilities
rf_test_scores = rf_pipeline.predict_proba(X_test)[:, 1]

# 7. Evaluation Function
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# 8. Comparing Results
baseline_p50 = precision_at_k(baseline_test, y_test, 50)
rf_p50 = precision_at_k(rf_test_scores, y_test, 50)

results_df = pd.DataFrame({
    "Method": ["Week 4 Hand-Rule Baseline", "Random Forest Classifier"],
    "Precision@50": [f"{baseline_p50:.3f}", f"{rf_p50:.3f}"]
})

print("--- Model vs. Baseline (Unseen Clients) ---")
display(results_df)


--- Model vs. Baseline (Unseen Clients) ---


,Method,Precision@50
0,Week 4 Hand-Rule Baseline,0.620
1,Random Forest Classifier,0.560


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

What the model leans on:
Extracting feature importances shows that the Random Forest heavily relies on content_age_days, impressions_90d, and avg_position. Unlike the flat baseline, it penalizes pages with extremely low impressions (noise) and heavily weighs the interaction between a page's age and its current SERP position.  

Where the model is wrong (Error Analysis):
Looking at the False Positives (pages ranked highly by the model that were not declining), the model tends to get fooled by seasonal content. For example, a page about "summer tires" might have high historical impressions, high age, and a decent average position, prompting the model to flag it as a high-risk decay candidate if we test it in autumn. The model lacks the contextual awareness of the calendar year to realize the drop in traffic is natural, not a semantic failure requiring a refresh.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Extracting and displaying feature importances
importances = rf_pipeline.named_steps["rf"].feature_importances_
importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("--- Feature Importances ---")
display(importance_df)


--- Feature Importances ---


,Feature,Importance
0,impressions_90d,0.362445
5,content_age_days,0.220926
2,avg_position,0.220748
4,word_count,0.096324
1,days_since_last_update,0.049878
3,ctr,0.049680


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.